# Dashboard final: siniestros viales

Este notebook integra los resultados del flujo de trabajo del TP para comunicar el problema, los hallazgos principales y el desempeno del modelo seleccionado. Consume el dataset procesado y los artefactos ya guardados en `outputs/`; para la matriz de confusion reconstruye el `RandomForestClassifier` ganador con el mismo split y parametros de evaluacion usados en los notebooks 06 y 07.

## Contexto del problema

El proyecto analiza registros de siniestros viales con el objetivo de comprender patrones de gravedad y evaluar si las caracteristicas de la victima y del contexto permiten anticipar casos graves o mortales. La comunicacion final busca combinar lectura descriptiva del fenomeno con evidencia del modelo predictivo.

## Hipotesis

La hipotesis de modelado plantea que las caracteristicas de la victima y del contexto del siniestro permiten anticipar si el caso terminara siendo grave o mortal. Para evitar leakage, el modelo excluye variables directamente asociadas al target o a la gravedad observada.

In [ ]:
from __future__ import annotations

import json
import logging
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

DATA_FILE = PROJECT_ROOT / "data" / "processed" / "siniestros_limpio_enriquecido.csv"
MODEL_METRICS_FILE = PROJECT_ROOT / "outputs" / "model_metrics.json"
MODEL_COMPARISON_FILE = PROJECT_ROOT / "outputs" / "model_comparison.json"
CROSS_VALIDATION_FILE = PROJECT_ROOT / "outputs" / "cross_validation_results.json"
DASHBOARD_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "dashboards"
FIGURES_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "figures"
DASHBOARD_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
FIGURES_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)
dashboard_path = DASHBOARD_OUTPUT_DIR / "dashboard_siniestros.html"
confusion_matrix_path = FIGURES_OUTPUT_DIR / "confusion_matrix_random_forest.png"
LOG_FILE = PROJECT_ROOT / "logs" / "pipeline.log"

LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
logger = logging.getLogger('dashboard_storytelling')
logger.setLevel(logging.INFO)
logger.handlers.clear()
handler = logging.FileHandler(LOG_FILE, mode='a', encoding='utf-8-sig')
handler.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(name)s | %(message)s'))
logger.addHandler(handler)
logger.info('Inicio de generacion del dashboard final de siniestros viales')

with MODEL_METRICS_FILE.open(encoding='utf-8') as file:
    model_metrics = json.load(file)
with MODEL_COMPARISON_FILE.open(encoding='utf-8') as file:
    model_comparison = json.load(file)
with CROSS_VALIDATION_FILE.open(encoding='utf-8') as file:
    cross_validation = json.load(file)

df = pd.read_csv(DATA_FILE)
df['fecha_siniestro'] = pd.to_datetime(df['fecha_siniestro'], errors='coerce')
df['gravedad_victima'] = df['gravedad_victima'].fillna('SIN_DATO')
df['edad_grupo'] = df['edad_grupo'].fillna('SIN_DATO')
df['vulnerabilidad_usuario'] = df['vulnerabilidad_usuario'].fillna('SIN_DATO')

gravedad_order = ['LEVE', 'GRAVE', 'MORTAL', 'SIN_DATO']
edad_order = ['menor_18', '18_30', '31_45', '46_60', 'mayor_60', 'SIN_DATO']

selected_model = cross_validation.get('best_model', model_comparison.get('best_model'))
best_f1 = cross_validation.get('best_model_metrics', {}).get('f1_mean', model_comparison.get('best_model_metrics', {}).get('f1'))
features = cross_validation.get('features', {})
feature_count = len(features.get('numeric', [])) + len(features.get('categorical', []))
severe_rate = df['es_grave_o_mortal'].mean() * 100

kpis = {
    'Registros totales': f'{len(df):,}'.replace(',', '.'),
    'Casos graves o mortales': f'{severe_rate:.2f}%',
    'Modelo seleccionado': selected_model,
    'Mejor F1 Score CV': f'{best_f1:.3f}',
    'Features usadas': str(feature_count),
}

def _nbformat_disponible():
    try:
        import nbformat
        version = tuple(int(part) for part in nbformat.__version__.split('.')[:2])
        return version >= (4, 2)
    except Exception:
        return False

def _mostrar_mensaje_render(mensaje):
    try:
        from IPython.display import HTML, display
        display(HTML(f"<p style='color:#6c757d'>{mensaje}</p>"))
    except Exception:
        print(mensaje)

PLOTLY_LIGHT_THEME = {
    "template": "plotly_white",
    "paper_bgcolor": "white",
    "plot_bgcolor": "white",
    "font": dict(
        family="Arial",
        size=14,
        color="black",
    ),
    "title_font": dict(
        size=20,
        color="black",
    ),
    "legend": dict(
        bgcolor="white",
        bordercolor="lightgray",
        borderwidth=1,
        font=dict(color="black"),
    ),
}

AXIS_LIGHT_THEME = dict(
    showgrid=True,
    gridcolor="lightgray",
    zerolinecolor="lightgray",
    color="black",
    title_font=dict(color="black"),
    tickfont=dict(color="black"),
)

def apply_plotly_light_theme(figure, height=520):
    figure.update_layout(**PLOTLY_LIGHT_THEME)
    if figure.layout.height is None:
        figure.update_layout(height=height)
    figure.update_xaxes(**AXIS_LIGHT_THEME)
    figure.update_yaxes(**AXIS_LIGHT_THEME)
    for annotation in figure.layout.annotations:
        annotation.font = dict(color="black")
    return figure

def show_fig(figure):
    if 'ipykernel' not in sys.modules:
        return

    if _nbformat_disponible():
        try:
            figure.show()
            return
        except Exception as exc:
            render_error = exc
    else:
        render_error = RuntimeError('nbformat>=4.2.0 no esta disponible para el render MIME de Plotly')

    try:
        fallback_path = DASHBOARD_OUTPUT_DIR / f"plotly_render_fallback_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.html"
        figure.write_html(fallback_path, include_plotlyjs='cdn', full_html=True)
        mensaje = f"No se pudo renderizar la figura en el notebook. Se exporto un HTML temporal en: {fallback_path}"
        logger.warning('%s Error original: %s', mensaje, render_error)
        _mostrar_mensaje_render(mensaje)
    except Exception as html_error:
        mensaje = f"No se pudo renderizar ni exportar la figura de Plotly. Error original: {render_error}. Error HTML: {html_error}"
        logger.warning(mensaje)
        _mostrar_mensaje_render(mensaje)

logger.info('Datos y artefactos cargados para dashboard: filas=%s, modelo=%s', len(df), selected_model)
print(kpis)

## KPIs principales

Los KPIs resumen la escala del dataset, la proporcion de casos graves o mortales, el modelo elegido, su mejor F1 promedio de validacion cruzada y la cantidad de variables predictoras usadas.

In [ ]:
kpi_fig = make_subplots(rows=1, cols=5, specs=[[{'type': 'indicator'} for _ in range(5)]])
for index, (title, value) in enumerate(kpis.items(), start=1):
    mode = 'number' if title in {'Registros totales', 'Features usadas'} else 'number'
    display_value = value
    if title == 'Registros totales':
        numeric_value = len(df)
        suffix = ''
    elif title == 'Features usadas':
        numeric_value = feature_count
        suffix = ''
    elif title == 'Casos graves o mortales':
        numeric_value = severe_rate
        suffix = '%'
    elif title == 'Mejor F1 Score CV':
        numeric_value = best_f1
        suffix = ''
    else:
        numeric_value = 0
        suffix = ''
    if title == 'Modelo seleccionado':
        kpi_fig.add_trace(go.Indicator(mode='number', value=0, title={'text': f'<b>{title}</b><br><span style="font-size:20px">{display_value}</span>'}, number={'font': {'size': 1}, 'valueformat': ' '}), row=1, col=index)
    else:
        kpi_fig.add_trace(go.Indicator(mode=mode, value=numeric_value, title={'text': f'<b>{title}</b>'}, number={'suffix': suffix}), row=1, col=index)

kpi_fig.update_layout(height=220, margin=dict(l=20, r=20, t=30, b=20))
apply_plotly_light_theme(kpi_fig)
show_fig(kpi_fig)

## Visualizaciones descriptivas y predictivas

Las visualizaciones combinan lectura del dataset procesado con resultados del modelo: distribucion de gravedad, perfiles de mayor riesgo, evolucion temporal, comparacion entre modelos y estabilidad por Cross Validation.

In [ ]:
gravedad_counts = (
    df['gravedad_victima']
    .value_counts()
    .reindex([value for value in gravedad_order if value in df['gravedad_victima'].unique()])
    .reset_index()
)
gravedad_counts.columns = ['gravedad_victima', 'cantidad']
gravedad_counts['porcentaje'] = gravedad_counts['cantidad'] / gravedad_counts['cantidad'].sum() * 100

fig_gravedad = px.bar(
    gravedad_counts,
    x='gravedad_victima',
    y='cantidad',
    text=gravedad_counts['porcentaje'].map(lambda value: f'{value:.1f}%'),
    color='gravedad_victima',
    color_discrete_map={'LEVE': '#2E86AB', 'GRAVE': '#F18F01', 'MORTAL': '#C73E1D', 'SIN_DATO': '#6C757D'},
    title='Distribucion de gravedad de las victimas',
)
fig_gravedad.update_layout(showlegend=False, xaxis_title='Gravedad', yaxis_title='Cantidad de registros')
fig_gravedad.update_traces(textposition='outside')
apply_plotly_light_theme(fig_gravedad)
show_fig(fig_gravedad)

edad_gravedad = (
    df.groupby(['edad_grupo', 'gravedad_victima'])
    .size()
    .reset_index(name='cantidad')
)
edad_gravedad['total_grupo'] = edad_gravedad.groupby('edad_grupo')['cantidad'].transform('sum')
edad_gravedad['porcentaje'] = edad_gravedad['cantidad'] / edad_gravedad['total_grupo'] * 100
edad_gravedad['edad_grupo'] = pd.Categorical(edad_gravedad['edad_grupo'], categories=edad_order, ordered=True)
edad_gravedad = edad_gravedad.sort_values('edad_grupo')

fig_edad = px.bar(
    edad_gravedad,
    x='edad_grupo',
    y='porcentaje',
    color='gravedad_victima',
    color_discrete_map={'LEVE': '#2E86AB', 'GRAVE': '#F18F01', 'MORTAL': '#C73E1D', 'SIN_DATO': '#6C757D'},
    title='Composicion de gravedad por grupo etario',
    labels={'porcentaje': 'Porcentaje dentro del grupo', 'edad_grupo': 'Grupo etario'},
)
fig_edad.update_layout(barmode='stack', yaxis_ticksuffix='%')
apply_plotly_light_theme(fig_edad)
show_fig(fig_edad)

vulnerabilidad_gravedad = (
    df.groupby(['vulnerabilidad_usuario', 'gravedad_victima'])
    .size()
    .reset_index(name='cantidad')
)
vulnerabilidad_gravedad['total_grupo'] = vulnerabilidad_gravedad.groupby('vulnerabilidad_usuario')['cantidad'].transform('sum')
vulnerabilidad_gravedad['porcentaje'] = vulnerabilidad_gravedad['cantidad'] / vulnerabilidad_gravedad['total_grupo'] * 100

fig_vulnerabilidad = px.bar(
    vulnerabilidad_gravedad,
    x='vulnerabilidad_usuario',
    y='porcentaje',
    color='gravedad_victima',
    color_discrete_map={'LEVE': '#2E86AB', 'GRAVE': '#F18F01', 'MORTAL': '#C73E1D', 'SIN_DATO': '#6C757D'},
    title='Composicion de gravedad por vulnerabilidad del usuario',
    labels={'porcentaje': 'Porcentaje dentro del grupo', 'vulnerabilidad_usuario': 'Vulnerabilidad'},
)
fig_vulnerabilidad.update_layout(barmode='stack', yaxis_ticksuffix='%')
apply_plotly_light_theme(fig_vulnerabilidad)
show_fig(fig_vulnerabilidad)

temporal = (
    df.dropna(subset=['fecha_siniestro'])
    .assign(periodo=lambda data: data['fecha_siniestro'].dt.to_period('M').dt.to_timestamp())
    .groupby(['periodo', 'es_grave_o_mortal'])
    .size()
    .reset_index(name='cantidad')
)
temporal['resultado'] = temporal['es_grave_o_mortal'].map({0: 'Leve', 1: 'Grave o mortal'})
fig_temporal = px.line(
    temporal,
    x='periodo',
    y='cantidad',
    color='resultado',
    markers=True,
    title='Evolucion temporal mensual de siniestros',
    labels={'periodo': 'Mes', 'cantidad': 'Cantidad de registros', 'resultado': 'Resultado'},
    color_discrete_map={'Leve': '#2E86AB', 'Grave o mortal': '#C73E1D'},
)
apply_plotly_light_theme(fig_temporal)
show_fig(fig_temporal)

comparison = pd.DataFrame(model_comparison.get('comparison_table', []))
comparison_long = comparison.melt(id_vars='modelo', value_vars=['accuracy', 'precision', 'recall', 'f1'], var_name='metrica', value_name='valor')
fig_modelos = px.bar(
    comparison_long,
    x='modelo',
    y='valor',
    color='metrica',
    barmode='group',
    title='Comparacion de metricas en holdout por modelo',
    labels={'modelo': 'Modelo', 'valor': 'Valor', 'metrica': 'Metrica'},
    color_discrete_sequence=['#2E86AB', '#6A4C93', '#F18F01', '#C73E1D'],
)
fig_modelos.update_layout(yaxis_range=[0, 1])
apply_plotly_light_theme(fig_modelos)
show_fig(fig_modelos)

cv_table = pd.DataFrame(cross_validation.get('comparison_table', []))
fig_cv = px.bar(
    cv_table,
    x='modelo',
    y='f1_mean',
    error_y='f1_std',
    color='modelo',
    title='F1 promedio en Cross Validation con desvio estandar',
    labels={'modelo': 'Modelo', 'f1_mean': 'F1 promedio', 'f1_std': 'Desvio estandar'},
    color_discrete_sequence=['#2E86AB', '#F18F01', '#C73E1D'],
)
fig_cv.update_layout(showlegend=False, yaxis_range=[0, max(0.35, cv_table['f1_mean'].max() + 0.05)])
apply_plotly_light_theme(fig_cv)
show_fig(fig_cv)

## Hallazgos principales

- La gran mayoria de los registros corresponde a casos leves, por lo que el problema predictivo esta desbalanceado.
- La proporcion de casos graves o mortales es baja, pero analiticamente relevante porque representa el evento de mayor impacto.
- Las variables de edad, modo de desplazamiento, rol y vulnerabilidad permiten construir una lectura de riesgo sin usar variables que filtran directamente la gravedad observada.
- La comparacion de modelos muestra que el desempeno no debe leerse solo por accuracy: en un dataset desbalanceado, recall y F1 son mas informativos para detectar casos graves o mortales.

## Resultado del modelo

El modelo seleccionado es el de mejor desempeno segun la estrategia de validacion definida. La decision prioriza F1 promedio y estabilidad entre folds, buscando un equilibrio entre deteccion de casos positivos y control de falsos positivos. El resultado es adecuado para un trabajo exploratorio y de trazabilidad academica, pero no debe interpretarse como un sistema listo para decision operativa sin nuevas validaciones.

## Evaluacion detallada del modelo ganador

La matriz de confusion permite analizar el comportamiento del `RandomForestClassifier` mas alla de una metrica agregada. En este problema la clase positiva representa casos graves o mortales, por lo que interesa distinguir cuantos casos positivos fueron detectados y cuantos quedaron sin identificar.

### Interpretacion de la matriz de confusion

- **Verdaderos Negativos (TN):** casos no graves/mortales correctamente clasificados como no graves/mortales.
- **Falsos Positivos (FP):** casos no graves/mortales clasificados por el modelo como graves/mortales.
- **Falsos Negativos (FN):** casos graves/mortales clasificados por el modelo como no graves/mortales.
- **Verdaderos Positivos (TP):** casos graves/mortales correctamente detectados.

### Costo de los errores

Un falso positivo implica asignar alerta o prioridad a un caso que finalmente no era grave/mortal. Puede generar sobreestimacion del riesgo y uso adicional de recursos. Un falso negativo es mas costoso desde el punto de vista preventivo: significa no detectar un caso que si era grave/mortal, reduciendo la utilidad del modelo como herramienta de priorizacion.


In [ ]:
TARGET = model_comparison.get("target", "es_grave_o_mortal")
RANDOM_STATE = int(model_comparison.get("random_state", 42))
TEST_SIZE = float(model_comparison.get("test_size", 0.2))
winner_from_06 = model_comparison.get("best_model")
winner_from_07 = cross_validation.get("best_model")

if winner_from_06 != "RandomForestClassifier" or winner_from_07 != "RandomForestClassifier":
    logger.warning(
        "El modelo ganador esperado era RandomForestClassifier, pero 06=%s y 07=%s",
        winner_from_06,
        winner_from_07,
    )

numeric_features = features.get("numeric", [])
categorical_features = features.get("categorical", [])
selected_features = numeric_features + categorical_features
missing_required = [column for column in selected_features + [TARGET] if column not in df.columns]
if missing_required:
    raise KeyError(f"Faltan columnas requeridas para evaluar el modelo ganador: {missing_required}")

model_df = df[selected_features + [TARGET]].copy()
model_df[categorical_features] = model_df[categorical_features].fillna("SIN_DATO")
X = model_df[selected_features]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numeric", "passthrough", numeric_features),
    ]
)
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=RANDOM_STATE,
                class_weight="balanced",
                n_jobs=-1,
                min_samples_leaf=20,
            ),
        ),
    ]
)

random_forest_model.fit(X_train, y_train)
y_pred = random_forest_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()
rf_accuracy = accuracy_score(y_test, y_pred)
rf_precision = precision_score(y_test, y_pred, zero_division=0)
rf_recall = recall_score(y_test, y_pred, zero_division=0)
rf_f1 = f1_score(y_test, y_pred, zero_division=0)

artifact_cm = model_comparison.get("models", {}).get("RandomForestClassifier", {}).get("confusion_matrix")
if artifact_cm and artifact_cm != cm.tolist():
    logger.warning("La matriz recalculada difiere del artefacto guardado: calculada=%s artefacto=%s", cm.tolist(), artifact_cm)
else:
    logger.info("Matriz de confusion del RandomForest validada contra outputs/model_comparison.json")

confusion_labels = [[f"Verdaderos Negativos\n{tn}", f"Falsos Positivos\n{fp}"], [f"Falsos Negativos\n{fn}", f"Verdaderos Positivos\n{tp}"]]
plt.figure(figsize=(8, 6))
ax = sns.heatmap(
    cm,
    annot=confusion_labels,
    fmt="",
    cmap="Blues",
    cbar=False,
    xticklabels=["Predicho no grave/mortal", "Predicho grave/mortal"],
    yticklabels=["Real no grave/mortal", "Real grave/mortal"],
    linewidths=0.5,
    linecolor="white",
)
ax.set_title("Matriz de confusion - RandomForestClassifier")
ax.set_xlabel("Clase predicha")
ax.set_ylabel("Clase real")
plt.tight_layout()
plt.savefig(confusion_matrix_path, dpi=160, bbox_inches="tight")
plt.show()
logger.info(f"Matriz de confusion exportada en: {confusion_matrix_path}")

rf_metrics_summary = pd.DataFrame(
    [
        {
            "modelo": "RandomForestClassifier",
            "accuracy": rf_accuracy,
            "precision": rf_precision,
            "recall": rf_recall,
            "f1": rf_f1,
            "verdaderos_negativos": int(tn),
            "falsos_positivos": int(fp),
            "falsos_negativos": int(fn),
            "verdaderos_positivos": int(tp),
        }
    ]
)
display(rf_metrics_summary)

if rf_recall > rf_precision:
    confusion_conclusion = (
        "El Recall es mayor que la Precision: el modelo prioriza detectar casos graves o mortales, "
        "aunque eso implique generar mas falsos positivos. Esta conducta es razonable cuando el costo de no detectar "
        "un caso grave/mortal es mas alto que el costo de revisar alertas adicionales."
    )
else:
    confusion_conclusion = (
        "La Precision es mayor o igual que el Recall: el modelo es mas conservador al marcar casos graves o mortales, "
        "pero puede dejar una proporcion mayor de positivos reales sin detectar."
    )

print(confusion_conclusion)
logger.info(
    "Evaluacion RandomForest: accuracy=%.4f precision=%.4f recall=%.4f f1=%.4f tn=%s fp=%s fn=%s tp=%s",
    rf_accuracy,
    rf_precision,
    rf_recall,
    rf_f1,
    tn,
    fp,
    fn,
    tp,
)


## Limitaciones

- El target `es_grave_o_mortal` resume un fenomeno complejo en una variable binaria.
- Existe desbalance de clases, lo que limita la lectura de accuracy y exige priorizar metricas como recall y F1.
- Las predicciones disponibles se evaluan sobre artefactos ya generados; este dashboard no reentrena ni recalcula modelos.
- La calidad del modelo depende de la calidad del registro administrativo y de la consistencia de categorias como `SIN_DATO`.

## Mejoras futuras

- Incorporar variables geograficas, clima, hora del dia o infraestructura vial si estuvieran disponibles.
- Generar predicciones fila a fila persistidas para auditar errores y explicar casos individuales.
- Evaluar calibracion de probabilidades y curvas precision-recall.
- Probar tecnicas adicionales para clases desbalanceadas y validaciones temporales.

In [ ]:
figures = [kpi_fig, fig_gravedad, fig_edad, fig_vulnerabilidad, fig_temporal, fig_modelos, fig_cv]
confusion_matrix_relative_path = '../figures/confusion_matrix_random_forest.png'
sections = [
    ('Contexto', 'El dashboard resume el flujo final del TP: datos procesados, hallazgos descriptivos y desempeno predictivo.'),
    ('Hipotesis', cross_validation.get('hypothesis', model_metrics.get('hypothesis', ''))),
    ('Hallazgos principales', 'El evento grave o mortal es poco frecuente y el problema esta desbalanceado. Por eso, F1 y recall son mas utiles que accuracy para evaluar la capacidad del modelo de detectar casos de mayor impacto.'),
    ('Resultado del modelo', f"Modelo seleccionado: {selected_model}. Mejor F1 promedio de Cross Validation: {best_f1:.3f}. Features usadas: {feature_count}."),
    ('Costo de los errores', 'Un falso positivo indica un caso no grave/mortal marcado como grave/mortal. Un falso negativo indica un caso grave/mortal no detectado por el modelo, que es el error mas sensible en una herramienta de priorizacion preventiva.'),
    ('Conclusion de la matriz de confusion', confusion_conclusion),
    ('Limitaciones', 'El dashboard trabaja con el dataset procesado y artefactos existentes; para la matriz de confusion reconstruye el RandomForestClassifier ganador con la misma configuracion de evaluacion. El target binario simplifica la severidad real y la informacion depende de la calidad del registro administrativo.'),
    ('Mejoras futuras', 'Persistir predicciones fila a fila, incorporar nuevas variables contextuales, revisar calibracion y profundizar validaciones temporales.'),
]

kpi_cards = ''.join(
    f"""
    <article class='kpi-card'>
        <div class='kpi-label'>{label}</div>
        <div class='kpi-value'>{value}</div>
    </article>
    """
    for label, value in kpis.items()
)

plot_html = ''
for index, figure in enumerate(figures):
    plot_html += pio.to_html(figure, include_plotlyjs='cdn' if index == 0 else False, full_html=False)

confusion_matrix_html = f"""
<section>
    <h2>Matriz de confusion del modelo ganador</h2>
    <p>La matriz resume los aciertos y errores del RandomForestClassifier sobre el conjunto de test usado en la comparacion de modelos.</p>
    <img class='confusion-matrix' src='{confusion_matrix_relative_path}' alt='Matriz de confusion del RandomForestClassifier'>
    <table class='metrics-table'>
        <thead><tr><th>Accuracy</th><th>Precision</th><th>Recall</th><th>F1</th></tr></thead>
        <tbody><tr><td>{rf_accuracy:.3f}</td><td>{rf_precision:.3f}</td><td>{rf_recall:.3f}</td><td>{rf_f1:.3f}</td></tr></tbody>
    </table>
    <p>{confusion_conclusion}</p>
</section>
"""

story_html = ''.join(f"<section><h2>{title}</h2><p>{text}</p></section>" for title, text in sections)
created_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

html = f"""
<!doctype html>
<html lang='es'>
<head>
    <meta charset='utf-8'>
    <meta name='viewport' content='width=device-width, initial-scale=1'>
    <title>Dashboard final - Siniestros viales</title>
    <style>
        :root {{
            --ink: #17202a;
            --muted: #5c6773;
            --line: #d8dee6;
            --panel: #f7f9fb;
            --accent: #2e86ab;
        }}
        body {{
            margin: 0;
            font-family: Arial, Helvetica, sans-serif;
            color: var(--ink);
            background: #ffffff;
        }}
        header {{
            padding: 32px 48px 22px;
            border-bottom: 1px solid var(--line);
            background: var(--panel);
        }}
        main {{
            max-width: 1180px;
            margin: 0 auto;
            padding: 28px 28px 48px;
        }}
        h1 {{
            margin: 0 0 8px;
            font-size: 32px;
            line-height: 1.15;
        }}
        h2 {{
            margin: 26px 0 8px;
            font-size: 22px;
        }}
        p {{
            color: var(--muted);
            line-height: 1.55;
            max-width: 980px;
        }}
        .kpi-grid {{
            display: grid;
            grid-template-columns: repeat(5, minmax(150px, 1fr));
            gap: 12px;
            margin: 22px 0 16px;
        }}
        .kpi-card {{
            border: 1px solid var(--line);
            border-radius: 8px;
            padding: 16px;
            background: #fff;
        }}
        .kpi-label {{
            color: var(--muted);
            font-size: 13px;
            min-height: 34px;
        }}
        .kpi-value {{
            margin-top: 10px;
            font-size: 24px;
            font-weight: 700;
            color: var(--accent);
            overflow-wrap: anywhere;
        }}
        .plotly-graph-div {{
            margin: 18px 0 30px;
            border-top: 1px solid var(--line);
            padding-top: 14px;
        }}
        .confusion-matrix {{
            max-width: 760px;
            width: 100%;
            height: auto;
            display: block;
            margin: 18px 0;
            border: 1px solid var(--line);
        }}
        .metrics-table {{
            border-collapse: collapse;
            margin: 14px 0 22px;
            min-width: 420px;
        }}
        .metrics-table th,
        .metrics-table td {{
            border: 1px solid var(--line);
            padding: 10px 14px;
            text-align: right;
        }}
        .metrics-table th {{
            background: var(--panel);
        }}
        footer {{
            margin-top: 24px;
            padding-top: 18px;
            border-top: 1px solid var(--line);
            color: var(--muted);
            font-size: 13px;
        }}
        @media (max-width: 900px) {{
            header {{ padding: 24px; }}
            .kpi-grid {{ grid-template-columns: repeat(2, minmax(140px, 1fr)); }}
        }}
    </style>
</head>
<body>
    <header>
        <h1>Dashboard final: siniestros viales</h1>
        <p>Analisis descriptivo, comparacion de modelos y narrativa final para la defensa del TP.</p>
    </header>
    <main>
        <section class='kpi-grid'>{kpi_cards}</section>
        {story_html}
        {confusion_matrix_html}
        {plot_html}
        <footer>Generado el {created_at}. Fuente: dataset procesado y artefactos guardados en outputs/.</footer>
    </main>
</body>
</html>
"""

dashboard_path.write_text(html, encoding='utf-8')
logger.info(f'Dashboard exportado en: {dashboard_path}')
print(f'Dashboard generado: {dashboard_path}')